In [1]:
# import libraries
from pydantic import BaseModel
from openai import OpenAI
from openai import AsyncOpenAI
import asyncio
import random
from dotenv import load_dotenv
import os
import sys
from pathlib import Path
import json
import re
import sys

# set path to project root and import custom classes and functions
base_path = Path.cwd() / "../../../"
sys.path.append(str(base_path.resolve()))
from utils.evaluation import evaluate_seqeval, extract_spans, cross_span_evaluation

In [3]:
def create_llm_annotations(text, spans):

    # sort the spans according to their start and end index
    spans = sorted(spans, key=lambda x: x["start"])

    # store the text and set index variable
    llm_text = ""
    last_idx = 0

    # loop through spans and add the span with custom characters
    for span in spans:
        llm_text += text[last_idx:span["start"]]
        llm_text += f"@@{text[span["start"]:span["end"]]}##"
        last_idx = span["end"]

    # add the rest of the text
    llm_text += text[last_idx:]

    return llm_text

def llm_output_to_bio(annotated_text):

    # split words via a regex
    words = re.findall(r"@@.*?##|\w+|'\w+|[^\w\s]", annotated_text)

    # empty list to store the bio tags
    bio_tags = []

    # loop through all words
    for word in words:

        # if it is an annotated span, split words and assign bio labels
        if word.startswith("@@") and word.endswith("##"):
            entity_text = word[2:-2]
            entity_words = re.findall(r"\w+|'\w+|[^\w\s]", entity_text)
            for i, t in enumerate(entity_words):
                tag = "B-sg" if i == 0 else "I-sg"
                bio_tags.append((t, tag))
        else:
            # otherwise assign O tag
            bio_tags.append((word, "O"))

    return bio_tags

def add_special_characters(original_sentence, llm_list):

    matches = []

    # reorder the entities list according to length of the entity
    llm_list = sorted(llm_list, key=len, reverse=True)


    for entity in llm_list:
        # full-word match but allow possessive 's or ’s
        pattern = r"(?i)(?<![A-Za-z])(" + re.escape(entity) + r")(?:['’]s)?(?![A-Za-z])"

        for match in re.finditer(pattern, original_sentence):
            matches.append(match.span(1))

    
    non_overlapping = []
    occupied = [False] * len(original_sentence)

    for start, end in matches:
        if not any(occupied[start:end]):
            non_overlapping.append((start, end))
            for i in range(start, end):
                occupied[i] = True 

    non_overlapping.sort(key=lambda x: x[0])

    result = original_sentence
    for start, end in reversed(non_overlapping):
        result = result[:start] + "@@" + result[start:end] + "##" + result[end:]

    return result

In [4]:
# initialize empty dataset list
validation_data = []

with open("../../../01_data/classification/training_validation_sets/ner/validation_set.json", "r") as f:
    val_data = json.load(f)

# loop through all sentences in the data
for task in val_data:
    # get the sentence and all annotations
    text = task["sentence"]
    spans = task["annotations"]
    labels = [annotation["text"] for annotation in spans]
   
    llm_text = create_llm_annotations(text, spans)
    bio_tags = llm_output_to_bio(llm_text)

    # add everything to the dataset list
    validation_data.append({
        "text": text,
        "labels": labels,
        "llm_text": llm_text,
        "bio_tags": bio_tags
    })

In [5]:
# do an estimation of the costs for running inference with different models
model_costs = {
    "4o-mini":
    {"standard": {
        "input": 0.15,
        "output": 0.6
    },
    "batch": {
        "input": 0.075,
        "output": 0.3
    }},
    "4o":
     {"standard": {
        "input": 2.5,
        "output": 10
    },
    "batch": {
        "input": 1.25,
        "output": 5
    }},
    "5-nano": 
    {"standard": {
        "input": 0.05,
        "output": 0.4
    },
    "batch": {
        "input": 0.025,
        "output": 0.2
    }
}}

input_lengths = [1500, 2000, 3000]
token_multiple = 1.3
test_type = "Hyperparameter_Tuning"
gpt_mode = "standard"

print(f"{test_type} Phase with {gpt_mode} processing")
print("-"*70)
for model in model_costs:
    for input in input_lengths:
        if test_type == "Validation":
            input_costs = (input*token_multiple*len(validation_data)/1000000)*model_costs[model][gpt_mode]["input"]
            output_costs = (75*token_multiple*len(validation_data)/1000000)*model_costs[model][gpt_mode]["output"]
        elif test_type == "Hyperparameter_Tuning":
            input_costs = (4*input*token_multiple*len(validation_data)/1000000)*model_costs[model][gpt_mode]["input"]
            output_costs = (4*100*token_multiple*len(validation_data)/1000000)*model_costs[model][gpt_mode]["output"]
        elif test_type == "CV":
            input_costs = (input*token_multiple*5000/1000000)*model_costs[model][gpt_mode]["input"]
            output_costs = (75*token_multiple*5000/1000000)*model_costs[model][gpt_mode]["output"]
        elif test_type == "Inference":
            input_costs = (input*token_multiple*500000/1000000)*model_costs[model][gpt_mode]["input"]
            output_costs = (75*token_multiple*500000/1000000)*model_costs[model][gpt_mode]["output"]
        total_costs = input_costs + output_costs
        print(f"Total costs for {model} with input prompt of length {input}: {total_costs:.4f}")
    print("-"*70)

Hyperparameter_Tuning Phase with standard processing
----------------------------------------------------------------------
Total costs for 4o-mini with input prompt of length 1500: 1.4820
Total costs for 4o-mini with input prompt of length 2000: 1.8720
Total costs for 4o-mini with input prompt of length 3000: 2.6520
----------------------------------------------------------------------
Total costs for 4o with input prompt of length 1500: 24.7000
Total costs for 4o with input prompt of length 2000: 31.2000
Total costs for 4o with input prompt of length 3000: 44.2000
----------------------------------------------------------------------
Total costs for 5-nano with input prompt of length 1500: 0.5980
Total costs for 5-nano with input prompt of length 2000: 0.7280
Total costs for 5-nano with input prompt of length 3000: 0.9880
----------------------------------------------------------------------


In [6]:
medium = """
## Task Objective
Extract all qualifying social groups from the sentence as a JSON object. Do not paraphrase the mentions at all! If no social groups are present, return an empty JSON array.

## Definition of a Social Group
A social group is a collective of people sharing **socio-demographic attributes** (age, ethnicity, income, occupation etc.).

**Do not** extract:
- institutional groups, political groups and state authorities ("small businesses", "armed forces", "Government", "Ministers")
- individual persons or highly specific collectives ("family of a colleague")

**Do** extract:
- subgroups of institutions with shared socio-demographic traits ("business people", "police officers")
- singular forms **only** if it is a generalization to a broader group ("every woman")
- social group component within a composite term ("victim support" -> "victim") or title ("Society for Disabled People" -> "Disabled People")
- additional sociodemographic description of the group ("women with mental health problems")
- extract as **one span** if several groups are mentioned and their meaning cannot be understood separately ("veterans, their wifes and children")
- general terms ("communities", "people") **only** if particular group is specified ("local communities", "people with special needs")

## Positive Examples:
- families, the taxpayer, victims, young people

## Negative Examples:
- EU, business, Tories, Labour, people, the public
"""

short = """
## Task Objective
Extract all qualifying social groups from the sentence as a JSON object. Do not paraphrase the mentions at all! If no social groups are present, return an empty JSON array.

## Definition of a Social Group
A social group is a collective of people sharing **socio-demographic attributes** (age, ethnicity, income, occupation etc.).

**Do not extract** institutional groups, firms, state authorities, groups based on political opinion or very specific small collectives such as one single family.
**Do extract** groupings of people within an institution and singular forms if it is generalizing to a broader collective.
Extract only the social group component unless there is an additional description to it.

## Positive Examples:
- families, the taxpayer, victims, criminals

## Negative Examples:
- EU, business, Tories, Labour, people, the public
"""

In [7]:
# compile manual few-shot examples in json format
positive_examples = [
    {"text": "We need to do more for young people, especially those with physical disabilities.",
     "llm_text": '{"social_groups": ["young people", "those with physical disabilities"]}'},
    {"text": "We want to support every child and ensure that the most vulnerable in this country can lead happy lives.",
     "llm_text": '{"social_groups": ["every child", "the most vulnerable in this country"]}'},
     {"text": "I want to express my support to all families in this country.",
     "llm_text": '{"social_groups": ["all families in this country"]}'}
]

negative_examples = [
    {"text": "The Government is working in close cooperaton with the Labour party on this matter.",
     "llm_text": '{"social_groups": []}'},
    {"text": "Businesses such as small- and medium-sized firms are vital for the economy in this country.",
     "llm_text": '{"social_groups": []}'},
     {"text": "Investment in the armed forces was certainly neglected under the previous Government.",
     "llm_text": '{"social_groups": []}'}
]

In [8]:
# create prompt template for the entity recognition task
def compile_prompt_ner(system_prompt, positive_examples, negative_examples, test_sentence, num_few_shot=4):

        chat = [
                {
                        "role": "system",
                        "content": system_prompt
                }
        ]
        

        num_few_shot_each = num_few_shot//2

        for i in range(num_few_shot_each):

                # add a positive example
                chat.append({"role": "user", "content": f"Sentence: {positive_examples[i]['text']}"})
                chat.append({"role": "assistant", "content": positive_examples[i]["llm_text"]})

                # add a negative example
                chat.append({"role": "user", "content": f"Sentence: {negative_examples[i]['text']}"})
                chat.append({"role": "assistant", "content": negative_examples[i]["llm_text"]})
        
        # add the test sentence
        chat.append({"role": "user", "content": f"Sentence: {test_sentence}"})

        return chat

In [9]:
# create dictionary storing rate limits
model_limits = {
    "gpt-4o-mini":
    {"token_limit": 200000,
     "request_limit": 500
    },
    "gpt-4o":
    {"token_limit": 30000,
     "request_limit": 500
    },
    "gpt-5-nano":
    {"token_limit": 200000,
     "request_limit": 500
    }
    }


# function to send out the request
async def send_request(client, model_name, prompt, output_class, temp=None, reasoning_effort=None):

    if model_name == "gpt-5-nano" :                                   
        response = await client.responses.parse(model=model_name,
                                                input=prompt,
                                                text_format=output_class,
                                                reasoning = {"effort": reasoning_effort}
                                                )
    elif model_name == "gpt-4o-mini":
        response = await client.responses.parse(model=model_name,
                                                input=prompt,
                                                text_format=output_class,
                                                temperature=temp
                                                )

    response_list = response.output_parsed.social_group
    try:
        if not isinstance(response_list, list):
            return []
        if response_list == ['']:
            return []
        return response_list
    except Exception:
        return []
                                            
                                        
async def dispatch_all(client, model_name, system_message, positive_examples, negative_examples,
                       num_fs, output_class, validation_data, temp, reasoning_effort, safe_interval):
    tasks = []
    for row in validation_data:
        sentence = row["text"]
        prompt = compile_prompt_ner(
            system_message,
            positive_examples,
            negative_examples,
            sentence,
            num_few_shot=num_fs
        )
        # create a task and fire it, do not wait
        task = asyncio.create_task(send_request(client, model_name, prompt, output_class, temp, reasoning_effort))
        tasks.append(task)
        
        # wait before starting the next request
        await asyncio.sleep(safe_interval)

    # gather all results once everything is started
    return await asyncio.gather(*tasks)

In [158]:
# select the model, system message and number of few shot examples
model_name = "gpt-5-nano"
system_message = medium
num_few_shot_examples = 4

# set temperature and reasoning effort
temp = 0
reasoning_effort = "medium"

# empirically test a safe rate per minute
if model_name == "gpt-4o":
    safe_rpm = 50
else:
    safe_rpm = 100

# calculate a safe interval in which requests are sent
safe_interval = 60.0 / safe_rpm

# define class for the output
class SocialGroupJSON(BaseModel):
    social_group: list[str]

# create client for interacting with API
load_dotenv()
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# get random indices
random.seed(7)
random_indices = random.sample(range(1000), 50)
val_subset = [validation_data[i] for i in random_indices]
llm_output = await dispatch_all(client, model_name, system_message, positive_examples, negative_examples,
                                num_few_shot_examples, SocialGroupJSON, val_subset, temp, reasoning_effort, safe_interval)

output_texts_special_characters = [add_special_characters(original["text"], llm_list) for original, llm_list in zip(val_subset, llm_output)]
output_bios = [llm_output_to_bio(text) for text in output_texts_special_characters]

In [107]:
# conduct manual error analysis
for idx in range(0, 50):
    print(val_subset[idx]["llm_text"])
    print(output_texts_special_characters[idx])
    print(llm_output[idx])
    print("-"*100)

As the hon. Gentleman knows, any proposals relating to any hospital must go through a proper process involving public and @@patient## engagement, sound clinical evidence, support by @@the GP commissioners##, and support for @@patient## choice.
As the hon. Gentleman knows, any proposals relating to any hospital must go through a proper process involving @@public## and patient engagement, sound clinical evidence, support by the @@GP commissioners##, and support for patient choice.
['public', 'GP commissioners']
----------------------------------------------------------------------------------------------------
The right hon. and learned Lady should stand up to her @@bosses## first.
The right hon. and learned Lady should stand up to her @@bosses## first.
['bosses']
----------------------------------------------------------------------------------------------------
That is higher than the 37 weeks admitted by Ministers and higher than the national average.
That is higher than the 37 weeks 

In [159]:
# evaluate the generated answers

# get list of bio tags only
ground_truth_bio = [[tag for (_, tag) in sent["bio_tags"]] for sent in val_subset]
pred_bio = [[tag for (_, tag) in sent] for sent in output_bios]

# evaluate at the entity level with seqeval
seqeval_results = evaluate_seqeval(ground_truth_bio, pred_bio)
print(f"Seqeval: {seqeval_results}")

# compute cross-span evaluation
all_true_spans = []
all_predicted_spans = []

for idx in range(len(ground_truth_bio)):
    # get the spans
    all_true_spans.append(extract_spans(ground_truth_bio[idx]))
    all_predicted_spans.append(extract_spans(pred_bio[idx]))

# apply cross-span evaluation
cross_span_results = cross_span_evaluation(all_true_spans, all_predicted_spans)
print(f"Cross-Span: {cross_span_results}")

Seqeval: {'precision': 0.7941176470588235, 'recall': 0.6585365853658537, 'f1': 0.72}
Cross-Span: {'precision': 0.7857142857142857, 'recall': 0.723015873015873, 'f1': 0.7428571428571429}


In [153]:
# conduct manual error analysis
for idx in range(0, 50):
    print(validation_data[idx]["llm_text"])
    print(output_texts_special_characters[idx])
    print(llm_output[idx])
    print("-"*100)

In addition, my @@officials## engage regularly with all interested @@stakeholders## to discuss gambling policy more generally, including the issue of fixed odds betting terminals.
In addition, my officials engage regularly with all interested stakeholders to discuss gambling policy more generally, including the issue of fixed odds betting terminals.
[]
----------------------------------------------------------------------------------------------------
I realise that our @@fellow citizens in Ulster## have unfortunately had just as much experience of rioting as some of our British cities have.
I realise that our @@fellow citizens in Ulster## have unfortunately had just as much experience of rioting as some of our British cities have.
['fellow citizens in Ulster']
----------------------------------------------------------------------------------------------------
Twenty-one years after the death of Stephen Lawrence, reforms are needed so that those failures do not continue to cast a long 

In [160]:
# set up temperatures and top
temperatures = [0, 0.25, 0.5]
reasoning_efforts = [None, "low", "medium", "high"]

# create client for interacting with API
load_dotenv()
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# define class for the output
class SocialGroupJSON(BaseModel):
    social_group: list[str]

# tune temperature value for 4o-mini model
model_name = "gpt-4o-mini"
system_message = short
num_few_shot_examples = 4
safe_rpm = 100
safe_interval = 60.0 / safe_rpm
results_4o_mini = {}
print("Tuning of 4o-mini starts")
print("-"*50)
for temp in temperatures:
    print(f"Trial with temperature: {temp}")
    llm_output = await dispatch_all(client, model_name, system_message, positive_examples, negative_examples,
                                    num_few_shot_examples, SocialGroupJSON, validation_data, temp, None, safe_interval)
    output_texts_special_characters = [add_special_characters(original["text"], llm_list) for original, llm_list in zip(validation_data, llm_output)]
    output_bios = [llm_output_to_bio(text) for text in output_texts_special_characters]
    ground_truth_bio = [[tag for (_, tag) in sent["bio_tags"]] for sent in validation_data]
    pred_bio = [[tag for (_, tag) in sent] for sent in output_bios]
    filtered_gt_bio = [tag_list_gt for (tag_list_gt, tag_list_pred) in zip(ground_truth_bio, pred_bio) if len(tag_list_gt) == len(tag_list_pred)]
    filtered_pred_bio = [tag_list_pred for (tag_list_gt, tag_list_pred) in zip(ground_truth_bio, pred_bio) if len(tag_list_gt) == len(tag_list_pred)]
    report = evaluate_seqeval(filtered_gt_bio, filtered_pred_bio)
    print(f"Report for temperature {temp}: {report}")
    results_4o_mini[temp] = report

# test all reasoning effort values for 5-nano
model_name = "gpt-5-nano"
system_message = medium
num_few_shot_examples = 4
safe_rpm = 75
safe_interval = 60.0 / safe_rpm
results_5_nano = {}
print("-"*50)
print("Tuning of 5-nano starts")
print("-"*50)
for reasoning_effort in reasoning_efforts:
    print(f"Trial with reasoning effort: {reasoning_effort}")
    llm_output = await dispatch_all(client, model_name, system_message, positive_examples, negative_examples,
                                    num_few_shot_examples, SocialGroupJSON, validation_data, None, reasoning_effort, safe_interval)
    output_texts_special_characters = [add_special_characters(original["text"], llm_list) for original, llm_list in zip(validation_data, llm_output)]
    output_bios = [llm_output_to_bio(text) for text in output_texts_special_characters]
    ground_truth_bio = [[tag for (_, tag) in sent["bio_tags"]] for sent in validation_data]
    pred_bio = [[tag for (_, tag) in sent] for sent in output_bios]
    filtered_gt_bio = [tag_list_gt for (tag_list_gt, tag_list_pred) in zip(ground_truth_bio, pred_bio) if len(tag_list_gt) == len(tag_list_pred)]
    filtered_pred_bio = [tag_list_pred for (tag_list_gt, tag_list_pred) in zip(ground_truth_bio, pred_bio) if len(tag_list_gt) == len(tag_list_pred)]
    report = evaluate_seqeval(filtered_gt_bio, filtered_pred_bio)
    print(f"Report for reasoning effort {reasoning_effort}: {report}")
    results_5_nano[reasoning_effort] = report

--------------------------------------------------
Tuning of 5-nano starts
--------------------------------------------------
Trial with reasoning effort: None
Report for reasoning effort None: {'precision': 0.6995073891625616, 'recall': 0.5892116182572614, 'f1': 0.6396396396396398}
Trial with reasoning effort: low
Report for reasoning effort low: {'precision': 0.6770833333333334, 'recall': 0.5394190871369294, 'f1': 0.6004618937644342}
Trial with reasoning effort: medium
Report for reasoning effort medium: {'precision': 0.6823338735818476, 'recall': 0.5822959889349931, 'f1': 0.6283582089552239}
Trial with reasoning effort: high
Report for reasoning effort high: {'precision': 0.7041800643086816, 'recall': 0.6058091286307054, 'f1': 0.6513011152416357}


In [164]:
hyperparameter_tuning_results = [results_4o_mini, results_5_nano]
with open("hyperparameter_tuning_results/ht_genllm_ner.json", "w") as f:
    json.dump(hyperparameter_tuning_results, f)